In [2]:
#use environment.yml
import pandas as pd
import numpy as np
import http.client
import requests
import json
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import os
import holidays
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor
)
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path
from xgboost import XGBRegressor
import lightgbm as lgb
from scipy.sparse import csr_matrix

from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient


Prendre top 20stations

In [3]:
#Charger données
PATH = "../01_Data/"
dataset = pd.read_parquet(f"{PATH}dataset_top20_stations.parquet")

TOP20_STATION_LIST = PATH +'top20_station_list.csv'
top20_station= pd.read_csv(TOP20_STATION_LIST)

dataset.describe().T
dataset.info()
dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175200 entries, 0 to 175199
Data columns (total 23 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   station_id                 175200 non-null  object        
 1   date                       175200 non-null  datetime64[ns]
 2   date_hour                  175200 non-null  datetime64[ns]
 3   year                       175200 non-null  int32         
 4   month                      175200 non-null  int32         
 5   day                        175200 non-null  int32         
 6   jour_semaine               175200 non-null  object        
 7   hour                       175200 non-null  int64         
 8   num_bikes_taken            175200 non-null  int64         
 9   num_bikes_dropped          175200 non-null  int64         
 10  net_flow                   175200 non-null  int64         
 11  temp                       175200 non-null  float32 

,station_id,date,date_hour,year,month,day,jour_semaine,hour,num_bikes_taken,num_bikes_dropped,...,precipitation_total,average_wind_speed,coco,is_holiday,coco_label,coco_group,precipitation_total_round,precipitation_total_bin,temp_round,temp_bin
0,5374.01,2024-11-01,2024-11-01 00:00:00,2024,11,1,Vendredi,0,11,10,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
1,5374.01,2024-11-01,2024-11-01 01:00:00,2024,11,1,Vendredi,1,16,11,...,0.0,15.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",24.0,"(22.3, 24.75]"
2,5374.01,2024-11-01,2024-11-01 02:00:00,2024,11,1,Vendredi,2,10,5,...,0.0,7.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
3,5374.01,2024-11-01,2024-11-01 03:00:00,2024,11,1,Vendredi,3,4,3,...,0.0,22.700001,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",23.0,"(22.3, 24.75]"
4,5374.01,2024-11-01,2024-11-01 04:00:00,2024,11,1,Vendredi,4,1,3,...,0.0,11.000000,3,False,Nuageux - 3,Pas de pluie,0.0,"(-0.0105, 0.525]",22.0,"(19.85, 22.3]"


### Features engineering
Prédiction Random forest avec données temporelles -> apprentissage supervisé
Random Forest supoose que les observations sont indépendantes
-> création de lags: Utiliser les valeurs passées pour prédire la valeur future.
-> Fonctionnalités temporelles : Extraire des informations de la date (Jour de la semaine, mois, année, jour de l'année, trimestre, vacances).
-> Statistiques glissantes (Rolling statistics) : Calculer des moyennes ou des écarts-types sur une fenêtre de temps glissante (ex: moyenne des 7 derniers jours)



In [4]:
def prepare_features(dataset, month_studied= None):
    dataset = dataset.copy()
    
    dataset["date"] = pd.to_datetime(dataset["date"], format="%Y-%m-%d", errors="raise")
    dataset = dataset.sort_values(["station_id","date_hour"])
            
    # --- Filtrer les dates si nécessaire ---
    if month_studied is not None:           
        start = pd.to_datetime(month_studied[0] + "-01")
        end   = pd.to_datetime(month_studied[1] + "-01") + pd.offsets.MonthEnd(1)

        dataset = dataset[
            (dataset["date"] >= start) &
            (dataset["date"] <= end)
        ]
    # Target (t+1h) + time-series features
    dataset["y"] = dataset.groupby("station_id")["net_flow"].shift(-1)
    
    #Features temporelles supplémentaires
    dataset['hour_sin'] = np.sin(2*np.pi*dataset['hour']/24)
    dataset['hour_cos'] = np.cos(2*np.pi*dataset['hour']/24)
    dataset['day_sin'] = np.sin(2*np.pi*dataset['day']/31)
    dataset['day_cos'] = np.cos(2*np.pi*dataset['day']/31)
    dataset['month_sin'] = np.sin(2*np.pi*dataset['month']/12)
    dataset['month_cos'] = np.cos(2*np.pi*dataset['month']/12)
    dataset['is_weekend'] = dataset['jour_semaine'].isin(['Samedi','Dimanche']).astype(int)
    dataset['is_peak'] = (((dataset['hour']>=6) & (dataset['hour']<10)) | ((dataset['hour']>=16) & (dataset['hour']<20))).astype(int)
    
    
    # --- Features météo ---
    dataset['cold_weather'] = (dataset['temp']<5).astype(int)
    dataset['hot_weather'] = (dataset['temp']>30).astype(int)
    dataset['heavy_rain'] = (dataset['precipitation_total']>5).astype(int)
    
    # --- Lag features ---
    
    lags = [1,2,24]
    for l in lags:
        dataset[f"net_flow_lag_{l}"] = dataset.groupby("station_id")["net_flow"].shift(l)

    dataset["net_flow_roll_3"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(3).mean()
    )

    dataset["net_flow_roll_24"] = dataset.groupby("station_id")["net_flow"].transform(
        lambda x: x.shift(1).rolling(24).mean()
    )
        
    # optional lags for pickups/drops (if present)
    for col in ["num_bikes_taken", "num_bikes_dropped"]:
        if col in dataset.columns:
            dataset[f"{col}_lag_1"] = dataset.groupby("station_id")[col].shift(1)

    # --- Features finales pour le modèle ---
    features = [
        #'hour_sin','hour_cos','day_sin','day_cos','month_sin','month_cos',
        #'is_weekend','is_peak',
        "station_id", 'year', 'month', 'day', 'hour',
        'temp','precipitation_total','relative_humidity','average_wind_speed',
        #'cold_weather','hot_weather','heavy_rain',
        'num_bikes_taken_lag_1','num_bikes_dropped_lag_1',
        'net_flow_lag_1','net_flow_lag_2','net_flow_lag_24','net_flow_roll_3','net_flow_roll_24',
        'jour_semaine', 'coco_group', 'is_holiday', 'coco'
        
    ]
    
    # Drop lignes avec NaN issues des lags
    needed = ["y"] + [f"net_flow_lag_{l}" for l in lags] + ["net_flow_roll_3","net_flow_roll_24"]
    dataset = dataset.dropna(subset=needed).reset_index(drop=True)
    
    # Cible
    target = 'net_flow'
    
    cols_to_keep = [col for col in features if col in dataset.columns]
    if 'net_flow' not in cols_to_keep:
        cols_to_keep.append('net_flow')
        
    filtered_df = dataset[cols_to_keep].copy()
    
    return filtered_df, features, target


dataset_all, features, target = prepare_features(dataset)#, month_studied=month_studied)


In [5]:
FILE = "data/dataset_station_preprocessed.parquet"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

PATH="../data"
if FILE in os.listdir():
        os.remove(FILE)
dataset_all.to_parquet(FILE, index=False, compression="snappy")

dataset_fe = dataset_all[dataset_all["station_id"].isin(top20_station)].copy()

TOP20_FILE = "data/dataset_top20_stations.parquet"
if TOP20_FILE in os.listdir():
        os.remove(TOP20_FILE)
dataset_fe.to_parquet(TOP20_FILE, index=False, compression="snappy")

print("Nombre de lignes :", len(dataset_fe))

Nombre de lignes : 8735


Split train/test pour données temporelles:
- pas de validation croisée aléatoire (K-Fold classique). Il faut respecter la structure temporelle. Le jeu d'entrainement doit précéder le jeu de test

In [6]:
# Split chronologique
cut = int(len(dataset_fe) * 0.8)
train_df = dataset_fe.iloc[:cut].copy()
test_df  = dataset_fe.iloc[cut:].copy()

X_train = train_df[features]
y_train = train_df[target]
X_test  = test_df[features]
y_test  = test_df[target]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# Baselin
# FIX: shift(1) par station pour éviter la fuite entre stations
# net_flow(t) ≈ net_flow(t-1) — premier enregistrement de chaque station → 0
y_pred_baseline = (
    test_df
    .groupby("station_id", observed=True)["net_flow"]
    .shift(1)
    .fillna(0)
    .to_numpy()
)
rmse_baseline = float(np.sqrt(mean_squared_error(y_test, y_pred_baseline)))
mae_baseline  = float(mean_absolute_error(y_test, y_pred_baseline))
r2_baseline   = float(r2_score(y_test, y_pred_baseline))

#Preprocessing
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype) == "category"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocess, num_cols),
        ("cat", categorical_preprocess, cat_cols),
    ],
    remainder="drop"
)

Train shape: (6988, 20), Test shape: (1747, 20)


In [7]:
#Models
models = [
    ("LinearRegression", LinearRegression()),
    ("Ridge(alpha=1.0)", Ridge(alpha=1.0, random_state=42)),
    ("Lasso(alpha=0.001)", Lasso(alpha=0.001, random_state=42, max_iter=20000)),
    ("RandomForest(300,depth=14)", RandomForestRegressor(
        n_estimators=300, max_depth=14, random_state=42, n_jobs=-1
    )),
    ("ExtraTrees(500,depth=16)", ExtraTreesRegressor(
        n_estimators=500, max_depth=16, random_state=42, n_jobs=-1
    )),
    ("XGBoost", XGBRegressor(
        n_estimators=600, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.0, reg_lambda=1.0,
        random_state=42, n_jobs=-1
    )),
]

def score(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

In [8]:
#Train + score
results = []
X_train_prep = csr_matrix(preprocess.fit_transform(X_train))
X_test_prep  = csr_matrix(preprocess.transform(X_test))

for name, model in models:
    print(f"  Training {name}...")
    model.fit(X_train_prep, y_train)
    pred = model.predict(X_test_prep)
    rmse, mae, r2 = score(y_test, pred)
    results.append({
        "model":              name,
        "rmse":               rmse,
        "mae":                mae,
        "r2":                 r2,
        "rmse_gain_vs_baseline": rmse_baseline - rmse,
    })

#Tableau comparatif benchmark
rows_bench = [{
    "Modèle":        "Baseline (net_flow t-1)",
    "RMSE":          rmse_baseline,
    "MAE":           mae_baseline,
    "R²":            r2_baseline,
    "Gain vs Baseline": 0.0,
    "Bat le Baseline":  "—",
}]
for r in results:
    rows_bench.append({
        "Modèle":        r["model"],
        "RMSE":          r["rmse"],
        "MAE":           r["mae"],
        "R²":            r["r2"],
        "Gain vs Baseline": rmse_baseline - r["rmse"],
        "Bat le Baseline":  "✅" if r["rmse"] < rmse_baseline else "❌",
    })

df_bench = (
    pd.DataFrame(rows_bench)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
df_bench["RMSE"]          = df_bench["RMSE"].map("{:.4f}".format)
df_bench["MAE"]           = df_bench["MAE"].map("{:.4f}".format)
df_bench["R²"]            = df_bench["R²"].map("{:.4f}".format)
df_bench["Gain vs Baseline"] = df_bench["Gain vs Baseline"].map("{:+.4f}".format)

print("\n=== Benchmark — Tous les modèles (trié par RMSE) ===")
print(df_bench.to_string(index=False))

  Training LinearRegression...
  Training Ridge(alpha=1.0)...
  Training Lasso(alpha=0.001)...
  Training RandomForest(300,depth=14)...
  Training ExtraTrees(500,depth=16)...
  Training XGBoost...

=== Benchmark — Tous les modèles (trié par RMSE) ===
                    Modèle    RMSE    MAE      R² Gain vs Baseline Bat le Baseline
  ExtraTrees(500,depth=16)  6.2472 4.3322  0.5195          +4.2840               ✅
RandomForest(300,depth=14)  6.5490 4.5252  0.4720          +3.9821               ✅
                   XGBoost  6.5671 4.5646  0.4690          +3.9640               ✅
        Lasso(alpha=0.001)  7.6256 5.4594  0.2841          +2.9055               ✅
          Ridge(alpha=1.0)  7.6288 5.4650  0.2835          +2.9023               ✅
          LinearRegression  7.6294 5.4660  0.2834          +2.9018               ✅
   Baseline (net_flow t-1) 10.5311 7.4184 -0.3654          +0.0000               —


In [9]:
# Hyperparameter tuning ExtraTrees
et_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", ExtraTreesRegressor(
        random_state=42,
        n_jobs=1,
    ))
])

param_dist_et_pipe = {
    "model__n_estimators":  [300, 500, 700, 1000],
    "model__max_depth":     [10, 14, 18, 22],   # None = pas de limite
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf":  [1, 2, 4],
    "model__max_features":  ["sqrt", "log2", 0.5, 0.8],
    "model__bootstrap":     [True, False],             # False = comportement ExtraTrees natif
}

tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(
    estimator=et_pipe,
    param_distributions=param_dist_et_pipe,
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    verbose=1,
    n_jobs=2,             # parallélise les 250 fits CV
    random_state=42,
)

search.fit(X_train, y_train)

best_xgb  = search.best_estimator_
pred_best = best_xgb.predict(X_test)
rmse_best, mae_best, r2_best = score(y_test, pred_best)

#Tableau final
rows_final = [{
    "Modèle":           "Baseline (net_flow t-1)",
    "RMSE":             rmse_baseline,
    "MAE":              mae_baseline,
    "R²":               r2_baseline,
    "Gain vs Baseline": 0.0,
    "Bat le Baseline":  "—",
}]
for r in results:
    rows_final.append({
        "Modèle":           r["model"],
        "RMSE":             r["rmse"],
        "MAE":              r["mae"],
        "R²":               r["r2"],
        "Gain vs Baseline": rmse_baseline - r["rmse"],
        "Bat le Baseline":  "✅" if r["rmse"] < rmse_baseline else "❌",
    })
rows_final.append({
    "Modèle":           "ExtraTrees tuné (RandomizedSearchCV)",
    "RMSE":             rmse_best,
    "MAE":              mae_best,
    "R²":               r2_best,
    "Gain vs Baseline": rmse_baseline - rmse_best,
    "Bat le Baseline":  "✅" if rmse_best < rmse_baseline else "❌",
})

df_final = (
    pd.DataFrame(rows_final)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
df_final["RMSE"]             = df_final["RMSE"].map("{:.4f}".format)
df_final["MAE"]              = df_final["MAE"].map("{:.4f}".format)
df_final["R²"]               = df_final["R²"].map("{:.4f}".format)
df_final["Gain vs Baseline"] = df_final["Gain vs Baseline"].map("{:+.4f}".format)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")
print("\n=== Tableau final — Tous les modèles + ExtraTrees tuné (trié par RMSE) ===")
print(df_final.to_string(index=False))

Fitting 5 folds for each of 30 candidates, totalling 150 fits


/opt/anaconda3/envs/ml/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Best params: {'model__n_estimators': 700, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 0.8, 'model__max_depth': 14, 'model__bootstrap': False}
Best CV RMSE: 5.4420

=== Tableau final — Tous les modèles + ExtraTrees tuné (trié par RMSE) ===
                              Modèle    RMSE    MAE      R² Gain vs Baseline Bat le Baseline
            ExtraTrees(500,depth=16)  6.2472 4.3322  0.5195          +4.2840               ✅
ExtraTrees tuné (RandomizedSearchCV)  6.2521 4.3317  0.5188          +4.2790               ✅
          RandomForest(300,depth=14)  6.5490 4.5252  0.4720          +3.9821               ✅
                             XGBoost  6.5671 4.5646  0.4690          +3.9640               ✅
                  Lasso(alpha=0.001)  7.6256 5.4594  0.2841          +2.9055               ✅
                    Ridge(alpha=1.0)  7.6288 5.4650  0.2835          +2.9023               ✅
                    LinearRegression  7.6294 5.4660  0.2834          

In [10]:
param_dist_refined = {
    "model__n_estimators":      [500, 700, 900],
    "model__max_depth":         [14, 16, 18, 20],   # ← 16 ajouté
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf":  [1, 2],              # ← 1 ajouté (benchmark utilisait défaut=1)
    "model__max_features":      [0.8, "sqrt"],
    "model__bootstrap":         [False],             # ← fixé à False (meilleur résultat)
}

search2 = RandomizedSearchCV(
    estimator=et_pipe,
    param_distributions=param_dist_refined,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=tscv,
    verbose=1,
    n_jobs=1,
    random_state=42,
)
search2.fit(X_train, y_train)

best_et2  = search2.best_estimator_
pred_best2 = best_et2.predict(X_test)
rmse_best2, mae_best2, r2_best2 = score(y_test, pred_best2)

print(f"Best params: {search2.best_params_}")
print(f"Best CV RMSE: {-search2.best_score_:.4f}")
print(f"TEST — RMSE: {rmse_best2:.4f} | MAE: {mae_best2:.4f} | R²: {r2_best2:.4f}")

with open("model/best_params.json", "w") as f:
    json.dump(search2.best_params_, f, indent=2)
print("best_params sauvegardés")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params: {'model__n_estimators': 900, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 0.8, 'model__max_depth': 20, 'model__bootstrap': False}
Best CV RMSE: 5.4332
TEST — RMSE: 6.2324 | MAE: 4.3226 | R²: 0.5218
best_params sauvegardés


### Sauvegarde modèle

In [11]:
final_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("model", ExtraTreesRegressor(
        n_estimators=search2.best_params_['model__n_estimators'],
        max_depth=search2.best_params_['model__max_depth'],
        min_samples_split=search2.best_params_['model__min_samples_split'],
        min_samples_leaf=search2.best_params_['model__min_samples_leaf'],
        max_features=search2.best_params_['model__max_features'],
        bootstrap=search2.best_params_['model__bootstrap'],
        random_state=42,
        n_jobs=-1,    # safe ici — plus de RandomizedSearchCV au-dessus
    ))
])
 
final_pipe.fit(X_train, y_train)
 
# Vérification finale sur le test set
pred_final = final_pipe.predict(X_test)
rmse_f, mae_f, r2_f = score(y_test, pred_final)

print(f"Pipeline final — RMSE: {rmse_f:.4f} | MAE: {mae_f:.4f} | R²: {r2_f:.4f}")
 
#Sauvegarde locale
Path("model").mkdir(exist_ok=True)
model_path = "model/citibike_forecast_model.joblib"
joblib.dump(final_pipe, model_path)
print("Pipeline sauvegardé localement")

# Vérifie la taille
size_mb = os.path.getsize(model_path) / 1e6
print(f"Pipeline sauvegardé — Taille : {size_mb:.1f} MB")


Pipeline final — RMSE: 6.2324 | MAE: 4.3226 | R²: 0.5218
Pipeline sauvegardé localement
Pipeline sauvegardé — Taille : 301.9 MB


In [12]:

# Modèle allégé — 200 arbres au lieu de 900
final_pipe_light = Pipeline(steps=[
    ("prep",  final_pipe.named_steps["prep"]),   # réutilise le preprocesseur déjà fitté
    ("model", ExtraTreesRegressor(
        n_estimators=200,        # 900 → 200
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=1,
        max_features=0.8,
        bootstrap=False,
        random_state=42,
        n_jobs=-1,
    ))
])
final_pipe_light.fit(X_train, y_train)

# Comparaison des perfs
pred_light = final_pipe_light.predict(X_test)
rmse_f_light, mae_f_light, r2_f_light = score(y_test, pred_light)

print(f"Pipeline final — RMSE: {rmse_f:.4f} | MAE: {mae_f:.4f} | R²: {r2_f:.4f}")
print(f"Pipeline final Light — RMSE: {rmse_f_light:.4f} | MAE: {mae_f_light:.4f} | R²: {r2_f_light:.4f}")

model_path_light = "model/citibike_forecast_model_light.joblib"
joblib.dump(final_pipe_light, model_path_light)
size_mb = os.path.getsize(model_path_light) / 1e6
print(f"Taille modèle light : {size_mb:.1f} MB")


Pipeline final — RMSE: 6.2324 | MAE: 4.3226 | R²: 0.5218
Pipeline final Light — RMSE: 6.2358 | MAE: 4.3311 | R²: 0.5213
Taille modèle light : 66.9 MB


In [17]:
os.environ["MLFLOW_TRACKING_URI"]    = "http://127.0.0.1:5001"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://127.0.0.1:9000"
os.environ["AWS_ACCESS_KEY_ID"]      = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"]  = "minioadmin"
os.environ["AWS_DEFAULT_REGION"]     = "us-east-1"
os.environ["NO_PROXY"]               = "127.0.0.1,localhost,minio"
os.environ["no_proxy"]               = "127.0.0.1,localhost,minio"
os.environ["HTTP_PROXY"]             = ""
os.environ["HTTPS_PROXY"]            = ""
os.environ["http_proxy"]             = ""
os.environ["https_proxy"]            = ""


# ── MLflow setup ──────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://127.0.0.1:5001"
EXPERIMENT_NAME     = "Citibike_forecast_training"
MODEL_NAME          = "citibike_forecast_model"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

exp = client.get_experiment_by_name(EXPERIMENT_NAME)
if exp and exp.lifecycle_stage == "deleted":
    client.restore_experiment(exp.experiment_id)
    experiment_id = exp.experiment_id
elif exp is None:
    experiment_id = client.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = exp.experiment_id

try:
    client.get_registered_model(MODEL_NAME)
except mlflow.exceptions.MlflowException:
    client.create_registered_model(MODEL_NAME)

# Charge les best_params sauvegardés
with open("model/best_params.json") as f:
    best_params = json.load(f)

# ── MLflow Run ────────────────────────────────────────────────────────────────
with mlflow.start_run(
    run_name="extratrees_light_200",
    experiment_id=experiment_id
) as run:
    rmse, mae, r2 = score(y_test, final_pipe_light.predict(X_test))

    mlflow.log_metric("rmse",               rmse)
    mlflow.log_metric("mae",                mae)
    mlflow.log_metric("r2",                 r2)
    mlflow.log_metric("rmse_baseline",      rmse_baseline)
    mlflow.log_metric("gain_vs_baseline",   rmse_baseline - rmse)
    mlflow.log_metric("n_estimators_full",  900)
    mlflow.log_metric("n_estimators_light", 200)
    mlflow.log_params(best_params)

    X_sig     = X_test.astype({col: "float64" for col in X_test.select_dtypes("int").columns})
    signature = infer_signature(X_sig, final_pipe_light.predict(X_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=final_pipe_light,
        artifact_path="model",
        signature=signature,
        registered_model_name=MODEL_NAME,
    )
    print(f"✅ Run ID : {run.info.run_id}")
    print(f"   RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}")

#Alias staging
versions = client.search_model_versions(
    filter_string=f"name='{MODEL_NAME}'",
    order_by=["version_number DESC"],
    max_results=1,
)
version = versions[0].version
client.set_registered_model_alias(name=MODEL_NAME, alias="staging", version=version)
print(f"✅ Version {version} aliasée 'staging'")
print(f"   Signature inputs  : {model_info.signature.inputs}")
print(f"   Signature outputs : {model_info.signature.outputs}")

Registered model 'citibike_forecast_model' already exists. Creating a new version of this model...
2026/05/10 12:17:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: citibike_forecast_model, version 2


✅ Run ID : 18cf023663f149d380fce1f7e3ce71ac
   RMSE=6.2358 | MAE=4.3311 | R²=0.5213
🏃 View run extratrees_light_200 at: http://127.0.0.1:5001/#/experiments/2/runs/18cf023663f149d380fce1f7e3ce71ac
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/2
✅ Version 2 aliasée 'staging'
   Signature inputs  : ['station_id': string (required), 'year': double (required), 'month': double (required), 'day': double (required), 'hour': double (required), 'temp': float (required), 'precipitation_total': float (required), 'relative_humidity': float (required), 'average_wind_speed': float (required), 'num_bikes_taken_lag_1': double (required), 'num_bikes_dropped_lag_1': double (required), 'net_flow_lag_1': double (required), 'net_flow_lag_2': double (required), 'net_flow_lag_24': double (required), 'net_flow_roll_3': double (required), 'net_flow_roll_24': double (required), 'jour_semaine': string (required), 'coco_group': string (required), 'is_holiday': boolean (required), 'coco': double (requir

Created version '2' of model 'citibike_forecast_model'.


### Conclusion — Modélisation CitiBike Flow Prediction

#### Résumé de la démarche

L'objectif était de prédire le **flux net de vélos** (`net_flow = vélos déposés - vélos pris`) par station et par heure, afin d'anticiper la disponibilité des vélos et des bornes sur le réseau CitiBike de New York.

La démarche a suivi 4 étapes :

1. **Baseline** — établir une référence simple : prédire le flux de l'heure précédente (`net_flow t-1`)
2. **Benchmark** — comparer 6 familles de modèles sur les mêmes données
3. **Sélection** — identifier le meilleur modèle (ExtraTrees)
4. **Tuning** — optimiser ses hyperparamètres en deux passes


#### Résultats finaux

| Modèle | RMSE | MAE | R² | Gain vs Baseline |
|---|---|---|---|---|
| **ExtraTrees tuné** ✅ | **6.23** | **4.32** | **0.52** | **+4.30** |
| ExtraTrees benchmark | 6.25 | 4.33 | 0.52 | +4.28 |
| RandomForest | 6.55 | 4.53 | 0.47 | +3.98 |
| XGBoost | 6.57 | 4.56 | 0.47 | +3.96 |
| Ridge / Lasso / LinReg | ~7.63 | ~5.46 | ~0.28 | +2.90 |
| **Baseline (net_flow t-1)** | **10.53** | **7.42** | **-0.37** | **0** |

Le modèle final réduit l'erreur de **+4.30 vélos par heure** par rapport au baseline, soit une réduction de **40% du RMSE**.

### Modèle retenu

**ExtraTrees Regressor** dans un `sklearn Pipeline` (imputation médiane + OHE) avec les hyperparamètres suivants :

| Hyperparamètre | Valeur | Justification |
|---|---|---|
| `n_estimators` | 900 | Plus d'arbres = meilleure stabilité |
| `max_depth` | 20 | Arbres profonds pour capturer les interactions lag × station × heure |
| `min_samples_leaf` | 1 | Feuilles fines — patterns locaux bien mémorisés |
| `min_samples_split` | 5 | Évite les splits sur trop peu d'observations |
| `max_features` | 0.8 | 80% des features par split — plus informatif que `sqrt` |
| `bootstrap` | False | Comportement ExtraTrees natif — plus performant ici |


### Limites du modèle

- **R² = 0.52** — le modèle explique 52% de la variance. Les 48% restants pourraient correspondre à des événements imprévisibles (incidents, événements sportifs, pannes de stations) non capturés dans les features
- **Modèle global** — un seul modèle pour toutes les stations. Un modèle par station pourrait capturer des patterns plus fins mais multiplierait la complexité de déploiement
- **Lag features uniquement** — le modèle ne dispose pas de données en temps réel sur l'état du réseau (vélos disponibles en ce moment dans les stations voisines)
- **Données météo horaires** — la météo est agrégée par heure ; une résolution plus fine (15 min) améliorerait probablement les prévisions sur les heures de pluie

### Perspectives

- Tester un **modèle par station** sur les 20 stations les plus actives
- Ajouter des **features de contexte réseau** : flux des stations voisines, nombre de vélos disponibles dans un rayon de 500m
- Explorer **LightGBM** — souvent plus rapide et parfois plus précis qu'ExtraTrees sur des datasets de cette taille
- Mettre en place une **réévaluation mensuelle automatique** du modèle via le DAG Airflow de monitoring déjà en place

*Modèle entraîné sur 6 988 observations (80%) · Évalué sur 1 747 observations (20%) · Split chronologique strict sans fuite temporelle*
